# 04 - BaseModel anidado para salida estructurada

Este notebook muestra un contrato Pydantic definido en el mismo notebook: un objeto principal con una lista de objetos hijos. Cada hijo mezcla texto y campos numericos.

In [1]:
import json

from pydantic import BaseModel, Field

from llmkit.config.settings import get_settings
from llmkit.llms import LLMFactory, build_messages

get_settings.cache_clear()
settings = get_settings()

MODEL_ID = settings.default_model
RUN_LIVE_LLM_CALLS = False

print("Model:", MODEL_ID)

Model: gpt-5.4-nano


## Contrato Pydantic

`ExperimentSummary` contiene una lista de `MetricObservation`. Esa forma aparece mucho en evaluacion: varias metricas, cada una con puntaje, peso y evidencia.

In [2]:
class MetricObservation(BaseModel):
    name: str = Field(description="Metric name, for example precision or latency.")
    value: float = Field(description="Measured numeric value.")
    target: float = Field(description="Expected or desired value.")
    passed: bool = Field(description="Whether the metric meets the target.")
    evidence: str = Field(description="Short text explaining the result.")

class ExperimentSummary(BaseModel):
    experiment_name: str
    overall_score: float = Field(ge=0, le=1)
    metrics: list[MetricObservation]
    recommendation: str

ExperimentSummary.model_json_schema()

{'$defs': {'MetricObservation': {'properties': {'name': {'description': 'Metric name, for example precision or latency.',
     'title': 'Name',
     'type': 'string'},
    'value': {'description': 'Measured numeric value.',
     'title': 'Value',
     'type': 'number'},
    'target': {'description': 'Expected or desired value.',
     'title': 'Target',
     'type': 'number'},
    'passed': {'description': 'Whether the metric meets the target.',
     'title': 'Passed',
     'type': 'boolean'},
    'evidence': {'description': 'Short text explaining the result.',
     'title': 'Evidence',
     'type': 'string'}},
   'required': ['name', 'value', 'target', 'passed', 'evidence'],
   'title': 'MetricObservation',
   'type': 'object'}},
 'properties': {'experiment_name': {'title': 'Experiment Name',
   'type': 'string'},
  'overall_score': {'maximum': 1,
   'minimum': 0,
   'title': 'Overall Score',
   'type': 'number'},
  'metrics': {'items': {'$ref': '#/$defs/MetricObservation'},
   'title'

## Prompt y messages

El modelo sigue recibiendo texto. Pydantic define como vamos a validar la respuesta despues.

In [3]:
system_prompt = (
    "You evaluate a small LLM experiment. Return only valid JSON matching the requested schema. "
    "Use concise evidence for each metric."
)

experiment_notes = """
Experiment: customer email classifier
Precision: 0.84, target: 0.80
Recall: 0.71, target: 0.75
Average latency seconds: 1.8, target: 2.0
The team cares most about precision before enabling automation.
""".strip()

user_prompt = (
    f"Notes:\n{experiment_notes}\n\n"
    "Return JSON with keys experiment_name, overall_score, metrics, and recommendation. "
    "Each metric must include name, value, target, passed, and evidence."
)

messages = build_messages(system_prompt, user_prompt)
messages

[{'role': 'system',
  'content': 'You evaluate a small LLM experiment. Return only valid JSON matching the requested schema. Use concise evidence for each metric.'},
 {'role': 'user',
  'content': 'Notes:\nExperiment: customer email classifier\nPrecision: 0.84, target: 0.80\nRecall: 0.71, target: 0.75\nAverage latency seconds: 1.8, target: 2.0\nThe team cares most about precision before enabling automation.\n\nReturn JSON with keys experiment_name, overall_score, metrics, and recommendation. Each metric must include name, value, target, passed, and evidence.'}]

## Ejecutar y validar

En modo demo usamos diccionarios locales validados con Pydantic. Si activas `RUN_LIVE_LLM_CALLS`, `invoke_structured()` envia el `BaseModel` al SDK de OpenAI para usar structured outputs reales.

In [4]:
if RUN_LIVE_LLM_CALLS:
    llm = LLMFactory.create(MODEL_ID)
    response = llm.invoke_structured(
        system=system_prompt,
        user=user_prompt,
        schema=ExperimentSummary,
    )
    summary = response.parsed
    print(response.content)
else:
    summary = ExperimentSummary.model_validate({
        "experiment_name": "customer email classifier",
        "overall_score": 0.78,
        "metrics": [
            {"name": "precision", "value": 0.84, "target": 0.80, "passed": True, "evidence": "Precision is above the automation threshold."},
            {"name": "recall", "value": 0.71, "target": 0.75, "passed": False, "evidence": "Some relevant customer emails are still missed."},
            {"name": "latency_seconds", "value": 1.8, "target": 2.0, "passed": True, "evidence": "Average response time is within the target."},
        ],
        "recommendation": "Use for assisted triage first; improve recall before full automation.",
    })
    print(summary.model_dump_json(indent=2))

summary

{
  "experiment_name": "customer email classifier",
  "overall_score": 0.78,
  "metrics": [
    {
      "name": "precision",
      "value": 0.84,
      "target": 0.8,
      "passed": true,
      "evidence": "Precision is above the automation threshold."
    },
    {
      "name": "recall",
      "value": 0.71,
      "target": 0.75,
      "passed": false,
      "evidence": "Some relevant customer emails are still missed."
    },
    {
      "name": "latency_seconds",
      "value": 1.8,
      "target": 2.0,
      "passed": true,
      "evidence": "Average response time is within the target."
    }
  ],
  "recommendation": "Use for assisted triage first; improve recall before full automation."
}


ExperimentSummary(experiment_name='customer email classifier', overall_score=0.78, metrics=[MetricObservation(name='precision', value=0.84, target=0.8, passed=True, evidence='Precision is above the automation threshold.'), MetricObservation(name='recall', value=0.71, target=0.75, passed=False, evidence='Some relevant customer emails are still missed.'), MetricObservation(name='latency_seconds', value=1.8, target=2.0, passed=True, evidence='Average response time is within the target.')], recommendation='Use for assisted triage first; improve recall before full automation.')

In [5]:
for metric in summary.metrics:
    status = "OK" if metric.passed else "REVIEW"
    print(f"{status:6} {metric.name:16} value={metric.value:.2f} target={metric.target:.2f} | {metric.evidence}")

print("Recommendation:", summary.recommendation)

OK     precision        value=0.84 target=0.80 | Precision is above the automation threshold.
REVIEW recall           value=0.71 target=0.75 | Some relevant customer emails are still missed.
OK     latency_seconds  value=1.80 target=2.00 | Average response time is within the target.
Recommendation: Use for assisted triage first; improve recall before full automation.


## JSONL: una lista de JSON por linea

Si tienes varios experimentos, un formato practico es `jsonl`: cada linea es un objeto JSON independiente. Eso permite correr el mismo prompt por registro y guardar una salida estructurada por fila.

In [6]:
jsonl_experiments = "\n".join([
    json.dumps({
        'experiment_name': 'customer email classifier',
        'notes': 'Precision: 0.84, target: 0.80\nRecall: 0.71, target: 0.75\nAverage latency seconds: 1.8, target: 2.0\nThe team cares most about precision before enabling automation.',
    }),
    json.dumps({
        'experiment_name': 'support ticket router',
        'notes': 'Precision: 0.78, target: 0.80\nRecall: 0.82, target: 0.78\nAverage latency seconds: 2.4, target: 2.0\nMisroutes create manual rework for the operations team.',
    }),
    json.dumps({
        'experiment_name': 'refund policy assistant',
        'notes': 'Precision: 0.91, target: 0.85\nRecall: 0.88, target: 0.80\nAverage latency seconds: 1.5, target: 2.0\nThe team needs a safe first candidate for partial automation.',
    }),
])

print(jsonl_experiments)
experiment_rows = [json.loads(line) for line in jsonl_experiments.splitlines() if line.strip()]
print("JSONL rows:", len(experiment_rows))
experiment_rows[0]

{"experiment_name": "customer email classifier", "notes": "Precision: 0.84, target: 0.80\nRecall: 0.71, target: 0.75\nAverage latency seconds: 1.8, target: 2.0\nThe team cares most about precision before enabling automation."}
{"experiment_name": "support ticket router", "notes": "Precision: 0.78, target: 0.80\nRecall: 0.82, target: 0.78\nAverage latency seconds: 2.4, target: 2.0\nMisroutes create manual rework for the operations team."}
{"experiment_name": "refund policy assistant", "notes": "Precision: 0.91, target: 0.85\nRecall: 0.88, target: 0.80\nAverage latency seconds: 1.5, target: 2.0\nThe team needs a safe first candidate for partial automation."}
JSONL rows: 3


{'experiment_name': 'customer email classifier',
 'notes': 'Precision: 0.84, target: 0.80\nRecall: 0.71, target: 0.75\nAverage latency seconds: 1.8, target: 2.0\nThe team cares most about precision before enabling automation.'}

In [7]:
first_row = experiment_rows[0]
batch_user_prompt = (
    f"Experiment: {first_row['experiment_name']}\n"
    f"Notes:\n{first_row['notes']}\n\n"
    "Return JSON with keys experiment_name, overall_score, metrics, and recommendation. "
    "Each metric must include name, value, target, passed, and evidence."
)

build_messages(system_prompt, batch_user_prompt)

[{'role': 'system',
  'content': 'You evaluate a small LLM experiment. Return only valid JSON matching the requested schema. Use concise evidence for each metric.'},
 {'role': 'user',
  'content': 'Experiment: customer email classifier\nNotes:\nPrecision: 0.84, target: 0.80\nRecall: 0.71, target: 0.75\nAverage latency seconds: 1.8, target: 2.0\nThe team cares most about precision before enabling automation.\n\nReturn JSON with keys experiment_name, overall_score, metrics, and recommendation. Each metric must include name, value, target, passed, and evidence.'}]

In [8]:
demo_batch_outputs = {
    'customer email classifier': {
        'experiment_name': 'customer email classifier',
        'overall_score': 0.78,
        'metrics': [
            {'name': 'precision', 'value': 0.84, 'target': 0.80, 'passed': True, 'evidence': 'Precision is above the automation threshold.'},
            {'name': 'recall', 'value': 0.71, 'target': 0.75, 'passed': False, 'evidence': 'Some relevant customer emails are still missed.'},
            {'name': 'latency_seconds', 'value': 1.8, 'target': 2.0, 'passed': True, 'evidence': 'Average response time is within the target.'},
        ],
        'recommendation': 'Use for assisted triage first; improve recall before full automation.',
    },
    'support ticket router': {
        'experiment_name': 'support ticket router',
        'overall_score': 0.73,
        'metrics': [
            {'name': 'precision', 'value': 0.78, 'target': 0.80, 'passed': False, 'evidence': 'Wrong routing still exceeds the acceptable limit.'},
            {'name': 'recall', 'value': 0.82, 'target': 0.78, 'passed': True, 'evidence': 'Most relevant tickets are captured.'},
            {'name': 'latency_seconds', 'value': 2.4, 'target': 2.0, 'passed': False, 'evidence': 'The workflow is slower than the routing target.'},
        ],
        'recommendation': 'Do not automate yet; reduce latency and routing errors first.',
    },
    'refund policy assistant': {
        'experiment_name': 'refund policy assistant',
        'overall_score': 0.89,
        'metrics': [
            {'name': 'precision', 'value': 0.91, 'target': 0.85, 'passed': True, 'evidence': 'High answer precision on policy questions.'},
            {'name': 'recall', 'value': 0.88, 'target': 0.80, 'passed': True, 'evidence': 'Most valid refund scenarios are covered.'},
            {'name': 'latency_seconds', 'value': 1.5, 'target': 2.0, 'passed': True, 'evidence': 'The assistant stays below the latency budget.'},
        ],
        'recommendation': 'Best candidate for a guarded rollout with human review.',
    },
}

batch_summaries = []
llm = LLMFactory.create(MODEL_ID) if RUN_LIVE_LLM_CALLS else None

for row in experiment_rows:
    row_user_prompt = (
        f"Experiment: {row['experiment_name']}\n"
        f"Notes:\n{row['notes']}\n\n"
        "Return JSON with keys experiment_name, overall_score, metrics, and recommendation. "
        "Each metric must include name, value, target, passed, and evidence."
    )

    if RUN_LIVE_LLM_CALLS:
        response = llm.invoke_structured(
            system=system_prompt,
            user=row_user_prompt,
            schema=ExperimentSummary,
        )
        batch_summaries.append(response.parsed)
    else:
        batch_summaries.append(ExperimentSummary.model_validate(demo_batch_outputs[row['experiment_name']]))

print('Parsed rows:', len(batch_summaries))
[summary.experiment_name for summary in batch_summaries]

Parsed rows: 3


['customer email classifier',
 'support ticket router',
 'refund policy assistant']

In [9]:
for summary in batch_summaries:
    passed_metrics = sum(metric.passed for metric in summary.metrics)
    print(f"{summary.experiment_name:26} score={summary.overall_score:.2f} passed_metrics={passed_metrics}/{len(summary.metrics)}")

customer email classifier  score=0.78 passed_metrics=2/3
support ticket router      score=0.73 passed_metrics=1/3
refund policy assistant    score=0.89 passed_metrics=3/3


## Segunda pasada: otro LLM resume o compara

Despues de obtener una salida estructurada por linea, puedes enviar la lista completa a otro prompt. Ese segundo paso sirve para resumir patrones comunes o comparar candidatos para despliegue.

`parse_json_output()` valida despues de generar texto. `invoke_structured()` manda el `BaseModel` al SDK como `response_format=Schema`, por lo que la forma esperada deja de depender solo del prompt. El error `summary` vs `summary_metrics` ocurre cuando se pide JSON generico en vez de structured output real.

In [10]:
class NamedMetricResult(BaseModel):
    name: str
    value: float
    target: float
    passed: bool

class BestExperiment(BaseModel):
    experiment_name: str
    overall_score: float = Field(ge=0, le=1)
    summary_metrics: list[NamedMetricResult]

class CommonFailure(BaseModel):
    issue: str
    experiments: list[str]
    impact: str
    example: str

class ExperimentOverview(BaseModel):
    name: str
    strengths: list[str]
    weaknesses: list[str]
    score_contributor: float = Field(ge=0, le=1)

class ComparisonSummary(BaseModel):
    best_candidate: str
    best_candidate_rationale: str
    experiment_overview: list[ExperimentOverview]
    overall_observations: str

class CrossExperimentComparison(BaseModel):
    best_experiment: BestExperiment
    common_failures: list[CommonFailure]
    comparison_summary: ComparisonSummary
    final_recommendation: str

CrossExperimentComparison.model_json_schema()

{'$defs': {'BestExperiment': {'properties': {'experiment_name': {'title': 'Experiment Name',
     'type': 'string'},
    'overall_score': {'maximum': 1,
     'minimum': 0,
     'title': 'Overall Score',
     'type': 'number'},
    'summary_metrics': {'items': {'$ref': '#/$defs/NamedMetricResult'},
     'title': 'Summary Metrics',
     'type': 'array'}},
   'required': ['experiment_name', 'overall_score', 'summary_metrics'],
   'title': 'BestExperiment',
   'type': 'object'},
  'CommonFailure': {'properties': {'issue': {'title': 'Issue',
     'type': 'string'},
    'experiments': {'items': {'type': 'string'},
     'title': 'Experiments',
     'type': 'array'},
    'impact': {'title': 'Impact', 'type': 'string'},
    'example': {'title': 'Example', 'type': 'string'}},
   'required': ['issue', 'experiments', 'impact', 'example'],
   'title': 'CommonFailure',
   'type': 'object'},
  'ComparisonSummary': {'properties': {'best_candidate': {'title': 'Best Candidate',
     'type': 'string'},
 

In [11]:
comparison_system_prompt = (
    "You compare multiple LLM experiment summaries. Return only valid JSON. "
    "Identify the best candidate, the common failures, and a concise deployment recommendation."
)

batch_payload = json.dumps(
    [summary.model_dump() for summary in batch_summaries],
    indent=2,
)

comparison_user_prompt = (
    f"Experiment summaries JSON:\n{batch_payload}\n\n"
    "Return JSON matching this contract: "
    "best_experiment is an object with experiment_name, overall_score, and summary_metrics; "
    "common_failures is a list of objects with issue, experiments, impact, and example; "
    "comparison_summary is an object with best_candidate, best_candidate_rationale, "
    "experiment_overview, and overall_observations; final_recommendation is a string."
)

comparison_messages = build_messages(comparison_system_prompt, comparison_user_prompt)
comparison_messages

[{'role': 'system',
  'content': 'You compare multiple LLM experiment summaries. Return only valid JSON. Identify the best candidate, the common failures, and a concise deployment recommendation.'},
 {'role': 'user',
  'content': 'Experiment summaries JSON:\n[\n  {\n    "experiment_name": "customer email classifier",\n    "overall_score": 0.78,\n    "metrics": [\n      {\n        "name": "precision",\n        "value": 0.84,\n        "target": 0.8,\n        "passed": true,\n        "evidence": "Precision is above the automation threshold."\n      },\n      {\n        "name": "recall",\n        "value": 0.71,\n        "target": 0.75,\n        "passed": false,\n        "evidence": "Some relevant customer emails are still missed."\n      },\n      {\n        "name": "latency_seconds",\n        "value": 1.8,\n        "target": 2.0,\n        "passed": true,\n        "evidence": "Average response time is within the target."\n      }\n    ],\n    "recommendation": "Use for assisted triage firs

In [12]:
if RUN_LIVE_LLM_CALLS:
    comparison_response = llm.invoke_structured(
        system=comparison_system_prompt,
        user=comparison_user_prompt,
        schema=CrossExperimentComparison,
    )
    comparison = comparison_response.parsed
    print(comparison_response.content)
else:
    comparison = CrossExperimentComparison.model_validate({
        'best_experiment': {
            'experiment_name': 'refund policy assistant',
            'overall_score': 1.0,
            'summary_metrics': [
                {'name': 'precision', 'value': 0.91, 'target': 0.85, 'passed': True},
                {'name': 'recall', 'value': 0.88, 'target': 0.80, 'passed': True},
                {'name': 'average_latency_seconds', 'value': 1.5, 'target': 2.0, 'passed': True},
            ],
        },
        'common_failures': [
            {
                'issue': 'Latency exceeding target',
                'experiments': ['support ticket router'],
                'impact': 'moderate',
                'example': '2.4s vs 2.0s target',
            },
            {
                'issue': 'Precision below target in at least one pilot',
                'experiments': ['support ticket router'],
                'impact': 'high risk of misrouting',
                'example': '0.78 vs 0.80',
            },
            {
                'issue': 'Recall below target in at least one pilot',
                'experiments': ['customer email classifier'],
                'impact': 'lower capture rate',
                'example': '0.71 vs 0.75',
            },
        ],
        'comparison_summary': {
            'best_candidate': 'refund policy assistant',
            'best_candidate_rationale': 'All metrics meet or exceed targets; strongest overall_score; fastest latency.',
            'experiment_overview': [
                {
                    'name': 'customer email classifier',
                    'strengths': ['precision above target (0.84)', 'latency within target (1.8s)'],
                    'weaknesses': ['recall below target (0.71)'],
                    'score_contributor': 0.7,
                },
                {
                    'name': 'support ticket router',
                    'strengths': ['recall above target (0.82)', 'latency close to target'],
                    'weaknesses': ['precision below target (0.78)', 'latency exceeded target (2.4s)'],
                    'score_contributor': 0.67,
                },
                {
                    'name': 'refund policy assistant',
                    'strengths': ['precision above target (0.91)', 'recall above target (0.88)', 'latency well below target (1.5s)'],
                    'weaknesses': [],
                    'score_contributor': 1.0,
                },
            ],
            'overall_observations': 'Tradeoffs between precision, recall, and latency across pilots. Deploy the strongest candidate first and keep improving the weaker pilots.',
        },
        'final_recommendation': 'Deploy the refund policy assistant first with guardrails and human review. Keep the other pilots in improvement mode until their missed metrics recover.',
    })
    print(comparison.model_dump_json(indent=2))

comparison

{
  "best_experiment": {
    "experiment_name": "refund policy assistant",
    "overall_score": 1.0,
    "summary_metrics": [
      {
        "name": "precision",
        "value": 0.91,
        "target": 0.85,
        "passed": true
      },
      {
        "name": "recall",
        "value": 0.88,
        "target": 0.8,
        "passed": true
      },
      {
        "name": "average_latency_seconds",
        "value": 1.5,
        "target": 2.0,
        "passed": true
      }
    ]
  },
  "common_failures": [
    {
      "issue": "Latency exceeding target",
      "experiments": [
        "support ticket router"
      ],
      "impact": "moderate",
      "example": "2.4s vs 2.0s target"
    },
    {
      "issue": "Precision below target in at least one pilot",
      "experiments": [
        "support ticket router"
      ],
      "impact": "high risk of misrouting",
      "example": "0.78 vs 0.80"
    },
    {
      "issue": "Recall below target in at least one pilot",
      "experiment

CrossExperimentComparison(best_experiment=BestExperiment(experiment_name='refund policy assistant', overall_score=1.0, summary_metrics=[NamedMetricResult(name='precision', value=0.91, target=0.85, passed=True), NamedMetricResult(name='recall', value=0.88, target=0.8, passed=True), NamedMetricResult(name='average_latency_seconds', value=1.5, target=2.0, passed=True)]), common_failures=[CommonFailure(issue='Latency exceeding target', experiments=['support ticket router'], impact='moderate', example='2.4s vs 2.0s target'), CommonFailure(issue='Precision below target in at least one pilot', experiments=['support ticket router'], impact='high risk of misrouting', example='0.78 vs 0.80'), CommonFailure(issue='Recall below target in at least one pilot', experiments=['customer email classifier'], impact='lower capture rate', example='0.71 vs 0.75')], comparison_summary=ComparisonSummary(best_candidate='refund policy assistant', best_candidate_rationale='All metrics meet or exceed targets; stro

In [13]:
print('Best experiment:', comparison.best_experiment.experiment_name)
print('Best rationale:', comparison.comparison_summary.best_candidate_rationale)
print('Common failures:')
for item in comparison.common_failures:
    print(f"- {item.issue} ({', '.join(item.experiments)}): {item.example}")
print('Recommendation:', comparison.final_recommendation)

Best experiment: refund policy assistant
Best rationale: All metrics meet or exceed targets; strongest overall_score; fastest latency.
Common failures:
- Latency exceeding target (support ticket router): 2.4s vs 2.0s target
- Precision below target in at least one pilot (support ticket router): 0.78 vs 0.80
- Recall below target in at least one pilot (customer email classifier): 0.71 vs 0.75
Recommendation: Deploy the refund policy assistant first with guardrails and human review. Keep the other pilots in improvement mode until their missed metrics recover.
